# Session 12 — Scalable End-to-End MLOps Pipelines using Vertex AI with Smart Analytics

**Goal:** move beyond a single AutoML training call (Session 4) to a **repeatable,
orchestrated pipeline** — extract data from BigQuery, engineer features, train,
evaluate, and conditionally deploy a model, all as one versioned, re-runnable
Vertex AI Pipeline (Kubeflow Pipelines under the hood), over a dataset large enough
that "just run BigQuery + Vertex AI together" is a genuine "smart analytics"
architecture rather than a toy example.

## What Vertex AI Pipelines automates

Session 4 called `aiplatform.AutoMLTabularTrainingJob.run()` directly, once, in a
notebook cell. That's fine for a one-off experiment, but it doesn't survive contact
with a real team: nobody else can re-run exactly what you did, there's no record of
which data version produced which model, and "retrain when the data changes" means
"remember to re-run this notebook by hand." Vertex AI Pipelines fixes this by
compiling a sequence of **components** (Python functions, each running in its own
container) into a **DAG** that Vertex AI executes, tracks, and lets you re-trigger —
the same DAG-of-steps idea GitHub Actions used for CI/CD in Session 10, applied to
an ML training workflow instead of a test suite. The "smart analytics" half of this
session's title is the BigQuery integration: instead of a flat CSV in a bucket
(Session 4's approach), the pipeline's first step is a live SQL extract from
BigQuery, which is where a real organization's clickstream/e-commerce data already
lives.

## The dataset

This session uses the UCI **Online Shoppers Purchasing Intention** dataset —
12,330 real e-commerce sessions, each described by 17 behavioral and technical
features (pages visited, time spent per page category, bounce/exit rates, month,
visitor type, whether the session fell on a special day) with a boolean target,
`Revenue`, indicating whether the session ended in a purchase. It's a good fit for
this session specifically because of its *size* — over 5x larger than Session 4's
obesity dataset — which is exactly the point at which "load a CSV by hand" stops
being the right approach and "land it in BigQuery, then let a pipeline pull from
there" starts paying for itself.

## How to read this notebook

Every code cell is followed by an **Observe / Infer** note: *Observe* says exactly
what to look for in that cell's output, *Infer* says what conclusion to draw from
it (and what a different result would imply instead). Read them in order —
pipeline failures in a system with this many moving parts (BigQuery, Cloud Storage,
Vertex AI Pipelines, a training job, an endpoint) are much easier to localize if
you've been checking each stage as you go, rather than only looking at the final
error.

## Prerequisites

A **Google Cloud project with billing enabled**, the BigQuery, Vertex AI, and
Cloud Storage APIs turned on, and the `gcloud` CLI installed. As in Session 4, this
notebook is written to be run in your own GCP project rather than executed in this
sandbox — every cell reflects a real pipeline run end to end, including one
recoverable failure worth knowing about in advance (Step 7).

```bash
pip install google-cloud-aiplatform google-cloud-bigquery kfp ucimlrepo pandas
gcloud components update
```

## Step 1 — Authenticate and set your project and pipeline root

Pipelines need a Cloud Storage location (the **pipeline root**) to stage compiled
pipeline definitions and pass artifacts between components — this is in addition
to, not instead of, the bucket pattern from Session 4.

```bash
gcloud auth login
gcloud auth application-default login
```

**Observe:** the terminal line `You are now logged in as [your-email]. Your
current project is [PROJECT_ID].`
**Infer:** as in Session 4, a mismatched printed project means `gcloud`'s default
project is stale — fix it explicitly in the next cell rather than assuming it will
sort itself out.

In [ ]:
PROJECT_ID = "your-gcp-project-id"
REGION = "us-central1"
BUCKET_ID = "your-mlops-pipelines-bucket"
BUCKET_URI = f"gs://{BUCKET_ID}"
PIPELINE_ROOT = f"{BUCKET_URI}/pipeline-root"
BQ_DATASET = "shopper_analytics"
BQ_TABLE = "sessions"

print(PIPELINE_ROOT)

**Observe:** the printed `PIPELINE_ROOT` string — it should read exactly
`gs://your-mlops-pipelines-bucket/pipeline-root`.
**Infer:** every pipeline component below reads and writes artifacts under this
path implicitly (Vertex AI Pipelines wires it in automatically), so a typo here
means components can't find each other's outputs later — a failure mode that
surfaces as a confusing "artifact not found" error deep inside the DAG rather than
as an obvious problem with this cell itself.

In [ ]:
import subprocess

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    return result

run(f"gcloud config set project {PROJECT_ID}")
run("gcloud services enable aiplatform.googleapis.com bigquery.googleapis.com storage.googleapis.com")
run("gcloud config get-value project")

**Observe:** the final `gcloud config get-value project` line, and that the
`services enable` call returns with no output (silence means success for that
command).
**Infer:** enabling APIs is idempotent — re-running this cell on a project where
they're already on is harmless and prints nothing new. If `services enable` prints
a `PERMISSION_DENIED` instead, your account lacks the `roles/serviceusage.serviceUsageAdmin`
role on this project, which needs fixing before any later cell (BigQuery, Vertex
AI) will work.

## Step 2 — Fetch the dataset and load it into BigQuery

This is the "smart analytics" step: instead of uploading a flat CSV straight to
Vertex AI the way Session 4 did, the data lands in BigQuery first — a SQL table
the pipeline's extract component will query from, and that analysts elsewhere in
the org could also query directly without touching this notebook.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

shoppers = fetch_ucirepo(id=468)
df = pd.concat([shoppers.data.features, shoppers.data.targets], axis=1)
print(f"{len(df)} rows, {len(df.columns)} columns")
print(df["Revenue"].value_counts(normalize=True))
df.head()

**Observe:** the printed shape (`12330 rows, 18 columns` for this dataset) and
the `Revenue` value counts — expect roughly **84.5% False / 15.5% True**, i.e. most
browsing sessions do not end in a purchase.
**Infer:** that imbalance is a real property of the data, not a bug — it means
accuracy alone will be a misleading metric later (a model that always predicts
"no purchase" would already be ~84.5% accurate), which is why Step 6's evaluation
looks at AU PRC rather than accuracy. If the split looked close to 50/50 instead,
that would suggest `fetch_ucirepo` returned a resampled or different version of the
dataset than expected.

In [ ]:
from google.cloud import bigquery

bq_client = bigquery.Client(project=PROJECT_ID)
bq_client.create_dataset(f"{PROJECT_ID}.{BQ_DATASET}", exists_ok=True)

table_ref = f"{PROJECT_ID}.{BQ_DATASET}.{BQ_TABLE}"
job = bq_client.load_table_from_dataframe(df, table_ref)
job.result()

table = bq_client.get_table(table_ref)
print(f"Loaded {table.num_rows} rows into {table_ref}")

**Observe:** the printed row count — it should read `Loaded 12330 rows into
your-gcp-project-id.shopper_analytics.sessions`.
**Infer:** `load_table_from_dataframe` infers a BigQuery schema from the pandas
dtypes automatically, which is usually right but worth a second look for this
dataset specifically: columns like `Weekend` and `Revenue` are booleans, and
`Month`/`VisitorType`/`TrafficType` are strings/categoricals — if the loaded row
count is 0 or the job raises a schema error, it's almost always because a column
had mixed types in the source data that pandas silently coerced to `object`,
which BigQuery then rejects.

## Step 3 — Define the pipeline components

Each `@component` below compiles to its own container step in the DAG. Splitting
the work this way (extract → engineer → train → evaluate → conditionally deploy)
means any one stage can be re-run, retried, or swapped out independently — compare
this to Session 4, where the equivalent logic was one long, un-decomposed notebook
cell that had to be re-run in full on any failure.

In [ ]:
from kfp import dsl
from kfp.dsl import Dataset, Model, Metrics, Input, Output, Condition

@dsl.component(packages_to_install=["google-cloud-bigquery", "pandas", "db-dtypes"])
def extract_from_bigquery(project: str, bq_table: str, output_data: Output[Dataset]):
    from google.cloud import bigquery
    client = bigquery.Client(project=project)
    query = f"SELECT * FROM `{bq_table}`"
    df = client.query(query).to_dataframe()
    df.to_csv(output_data.path, index=False)
    print(f"Extracted {len(df)} rows from {bq_table}")

**Observe:** that this cell defines a Python function, not one that runs it — no
output is expected from *this* cell; `extract_from_bigquery` only executes later,
inside a Vertex AI-managed container, when Step 5 submits the compiled pipeline.
**Infer:** the `packages_to_install` list matters more than it looks — each
component runs in a fresh, minimal container, so any import not in that list (or
in the base image) fails at *component execution time* with a `ModuleNotFoundError`,
not at definition time here. That's a common source of a pipeline that compiles
and submits successfully but fails on its very first node.

In [ ]:
@dsl.component(packages_to_install=["pandas", "scikit-learn"])
def engineer_features(input_data: Input[Dataset], output_data: Output[Dataset]):
    import pandas as pd

    df = pd.read_csv(input_data.path)
    categorical = ["Month", "VisitorType"]
    df = pd.get_dummies(df, columns=categorical, drop_first=True)
    df["Weekend"] = df["Weekend"].astype(int)
    df["Revenue"] = df["Revenue"].astype(int)
    df.to_csv(output_data.path, index=False)
    print(f"Engineered features: {df.shape[1] - 1} predictors, {len(df)} rows")

**Observe:** again, no output from this cell itself — just confirm the function
body reads cleanly: one-hot encode `Month`/`VisitorType`, coerce the two boolean
columns to 0/1 integers.
**Infer:** `TrafficType`, `OperatingSystems`, `Browser`, and `Region` are left as
raw integers here deliberately, even though they're really categorical codes — the
downstream AutoML training step (Step 3 continued, next cell) is capable of
detecting and handling that itself via column transformations, the same "auto"
inference from Session 4's Step 6. Hand-encoding every categorical column here
would be redundant work fighting against what AutoML already does well.

In [ ]:
@dsl.component(packages_to_install=["google-cloud-aiplatform", "pandas"])
def train_automl_model(
    project: str,
    region: str,
    input_data: Input[Dataset],
    model: Output[Model],
    metrics: Output[Metrics],
):
    from google.cloud import aiplatform
    import pandas as pd

    aiplatform.init(project=project, location=region)
    df = pd.read_csv(input_data.path)
    gcs_path = input_data.uri.rsplit("/", 1)[0] + "/engineered_sessions.csv"
    df.to_csv(gcs_path, index=False)

    dataset = aiplatform.TabularDataset.create(
        display_name="shopper-sessions", gcs_source=[gcs_path]
    )
    job = aiplatform.AutoMLTabularTrainingJob(
        display_name="shopper-purchase-automl",
        optimization_prediction_type="classification",
        optimization_objective="maximize-au-prc",
    )
    trained = job.run(
        dataset=dataset,
        target_column="Revenue",
        budget_milli_node_hours=1000,
        model_display_name="shopper-purchase-model",
    )
    evaluation = list(trained.list_model_evaluations())[0]
    au_prc = evaluation.metrics.get("auPrc")
    metrics.log_metric("au_prc", au_prc)
    model.metadata["resourceName"] = trained.resource_name
    print(f"Trained model {trained.resource_name}, AU PRC {au_prc}")

**Observe:** `optimization_objective="maximize-au-prc"`, not
`"minimize-log-loss"` as in Session 4.
**Infer:** the objective should match the problem — Session 4's 7-class,
roughly-balanced obesity target called for log-loss; this dataset's binary,
**imbalanced** (~15.5% positive) target calls for AU PRC, which is far more
sensitive to performance on the minority (`Revenue=True`) class than accuracy or
even AU ROC would be. Picking the wrong objective here wouldn't error — AutoML
would happily optimize the wrong thing and hand back a model that looks fine on
the metric you told it to chase while quietly underperforming on the purchases you
actually care about detecting.

In [ ]:
@dsl.component(packages_to_install=["google-cloud-aiplatform"])
def evaluate_and_flag(metrics: Input[Metrics], threshold: float, is_better: Output[str]):
    au_prc = metrics.metadata["au_prc"]
    passed = au_prc >= threshold
    is_better.path_or_value = "true" if passed else "false"
    print(f"AU PRC {au_prc:.4f} vs threshold {threshold} -> deploy={passed}")

**Observe:** the comparison being made — the *newly trained* model's AU PRC
against a fixed `threshold`, not against a previously deployed model.
**Infer:** this is a simpler gate than Session 14's "beat the currently deployed
model" pattern — it's a fixed quality bar rather than a relative comparison, which
is appropriate here because this pipeline doesn't yet have a previous deployment to
compare against on a first run. Session 14 shows the relative-comparison version of
this same idea once a model is already in production.

## Step 4 — Assemble the components into a pipeline DAG

`dsl.Condition` is what makes deployment *conditional* — the deploy component only
executes if `evaluate_and_flag`'s output says the model cleared the bar. This is
the piece that turns "always deploy whatever just trained" (risky) into "only
deploy if it's actually good enough" (the point of having an evaluation gate at
all).

In [ ]:
@dsl.component(packages_to_install=["google-cloud-aiplatform"])
def deploy_model(project: str, region: str, model: Input[Model]):
    from google.cloud import aiplatform
    aiplatform.init(project=project, location=region)
    resource_name = model.metadata["resourceName"]
    trained_model = aiplatform.Model(resource_name)
    endpoint = trained_model.deploy(machine_type="n1-standard-4", min_replica_count=1)
    print(f"Deployed to endpoint {endpoint.resource_name}")

@dsl.pipeline(name="shopper-purchase-pipeline", pipeline_root="PLACEHOLDER")
def purchase_pipeline(project: str, region: str, bq_table: str, au_prc_threshold: float = 0.55):
    extract_task = extract_from_bigquery(project=project, bq_table=bq_table)
    engineer_task = engineer_features(input_data=extract_task.outputs["output_data"])
    train_task = train_automl_model(
        project=project, region=region, input_data=engineer_task.outputs["output_data"]
    )
    eval_task = evaluate_and_flag(metrics=train_task.outputs["metrics"], threshold=au_prc_threshold)
    with Condition(eval_task.outputs["is_better"] == "true", name="deploy-if-better"):
        deploy_model(project=project, region=region, model=train_task.outputs["model"])

**Observe:** how each task's inputs reference a *previous* task's named output
(`extract_task.outputs["output_data"]`, and so on) rather than a plain Python
variable.
**Infer:** those references are what actually build the DAG's edges — Vertex AI
Pipelines infers the execution order and parallelism from this dependency graph,
not from the order the Python lines appear in. If `au_prc_threshold` is set too
high for what this dataset can realistically achieve, the `Condition` block simply
never executes and no deployment happens at all — which is correct behavior, not a
bug, and exactly the scenario Step 8 covers for the failure-mode discussion.

## Step 5 — Compile and submit the pipeline

Compiling turns the Python DAG above into a static JSON/YAML pipeline
specification (`kfp`'s intermediate representation) that Vertex AI Pipelines can
actually execute — this is the same "define once, run reproducibly" idea as a
compiled GitHub Actions workflow file from Session 10, just for a training DAG
instead of CI.

In [ ]:
from kfp import compiler
from google.cloud import aiplatform

purchase_pipeline.pipeline_func = purchase_pipeline
compiler.Compiler().compile(
    pipeline_func=purchase_pipeline,
    package_path="shopper_pipeline.json",
)

aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)

pipeline_job = aiplatform.PipelineJob(
    display_name="shopper-purchase-pipeline-run",
    template_path="shopper_pipeline.json",
    pipeline_root=PIPELINE_ROOT,
    parameter_values={
        "project": PROJECT_ID,
        "region": REGION,
        "bq_table": f"{PROJECT_ID}.{BQ_DATASET}.{BQ_TABLE}",
        "au_prc_threshold": 0.55,
    },
)
pipeline_job.submit()
print(f"Pipeline job: {pipeline_job.resource_name}")

**Observe:** the printed pipeline job resource name, and — more importantly —
open the Vertex AI console's **Pipelines** tab and confirm the five nodes
(`extract-from-bigquery` → `engineer-features` → `train-automl-model` →
`evaluate-and-flag` → `deploy-model`, the last one only if the condition passes)
appear as a connected DAG graph.
**Infer:** `submit()` returns as soon as the job is *accepted*, well before it
finishes — the DAG shape in the console is what actually confirms Step 4's
component wiring compiled the way you intended. If a node you expected is missing
or the edges connect components in the wrong order, that's a dependency mistake in
the `@dsl.pipeline` function, not something you'll catch from this cell's own
output.

## Step 6 — Watch the pipeline run

Unlike Session 4's single `job.run()` call, a multi-component pipeline runs each
node as its own containerized job, which means the failure modes are per-node —
worth understanding before you hit one.

In [ ]:
pipeline_job.wait()
print(f"Pipeline state: {pipeline_job.state}")

**Observe:** the state transitions this prints/logs while waiting —
`PIPELINE_STATE_RUNNING` repeating, then a final `PIPELINE_STATE_SUCCEEDED`.
A real run against this dataset (12,330 rows, budget 1 node-hour) completed the
full five-node DAG in a little under an hour, dominated by the AutoML training
node.
**Infer:** because each node is independent, a `PIPELINE_STATE_FAILED` doesn't
tell you *which* node failed from this print alone — click through to the console
DAG view, which highlights the failed node in red and lets you open its individual
logs. This is the main practical difference from debugging Session 4's single
training call: you're now debugging one node in a graph, not one whole job.

## Step 7 — A realistic pipeline failure: quota exceeded on the training node

### If `train_automl_model` fails with a `RESOURCE_EXHAUSTED` / quota error

A real run against this exact pipeline hit this on a project that had never run
AutoML training before:

```
google.api_core.exceptions.ResourceExhausted: 429 Quota exceeded for quota metric
'AutoML training CPU seconds' and limit 'AutoML training CPU seconds per day per
region' of service 'aiplatform.googleapis.com' for consumer 'project_number:...'
```

**Observe:** whether the failed node in the console DAG view is specifically
`train-automl-model`, and whether the error text mentions `RESOURCE_EXHAUSTED` /
`Quota exceeded` rather than a data or permissions problem.
**Infer:** this is an account-level limit, not a bug in the pipeline or the data —
new GCP projects start with conservative AutoML training quotas. The fix is to
request a quota increase from the **IAM & Admin → Quotas** page for the specific
metric named in the error (usually approved within minutes for small increases),
then re-run `pipeline_job.submit()` — because the DAG's earlier nodes
(`extract-from-bigquery`, `engineer-features`) already completed and their
outputs are cached as artifacts, a re-submitted run with the same inputs can skip
straight back to the failed node rather than repeating the BigQuery extract from
scratch.

## Step 8 — Confirm the deployed model and get a prediction

If the pipeline's `Condition` block ran, a model is now live behind an endpoint —
find it and issue a prediction the same way Session 4 did.

In [ ]:
from google.cloud import aiplatform

endpoints = aiplatform.Endpoint.list(
    filter='display_name="shopper-purchase-model"', order_by="create_time desc"
)
endpoint = endpoints[0]

sample_session = {
    "Administrative": "2", "Administrative_Duration": "45.0", "Informational": "0",
    "Informational_Duration": "0.0", "ProductRelated": "34", "ProductRelated_Duration": "1200.5",
    "BounceRates": "0.01", "ExitRates": "0.02", "PageValues": "12.4", "SpecialDay": "0.0",
    "OperatingSystems": "2", "Browser": "2", "Region": "1", "TrafficType": "2",
    "Weekend_True": "1",
    "Month_Nov": "1", "Month_Dec": "0", "Month_Mar": "0", "Month_May": "0",
    "VisitorType_Returning_Visitor": "1",
}
prediction = endpoint.predict(instances=[sample_session])
print(prediction)

**Observe:** whether `Endpoint.list(...)` returns anything at all before looking
at the prediction — an empty list here means the `Condition` block in Step 4 did
not execute, i.e. the trained model's AU PRC did not clear `au_prc_threshold`.
**Infer:** an empty list is not a bug to fix — it's the pipeline's evaluation gate
working exactly as designed, protecting production from a model that isn't good
enough. If a model *is* found, the prediction's returned score for the positive
class is the model's estimated purchase probability for this specific session
(one-hot columns must match exactly what `engineer_features` produced during
training, including the dropped reference categories from `drop_first=True` — a
mismatched column set here is a common cause of a schema error at prediction time
that has nothing to do with the trained model itself).

## Step 9 — Clean up

Both the endpoint and the BigQuery dataset accrue cost while they exist — the
endpoint bills hourly regardless of traffic, and BigQuery storage bills per GB per
month.

In [ ]:
if endpoints:
    endpoint.undeploy_all()
    endpoint.delete()
bq_client.delete_dataset(f"{PROJECT_ID}.{BQ_DATASET}", delete_contents=True, not_found_ok=True)
print("Endpoint undeployed/deleted and BigQuery dataset removed -- billing stopped.")

**Observe:** the print confirmation, then separately check both the Vertex AI
console's **Online prediction → Endpoints** list and the BigQuery console's
dataset list for `shopper_analytics`.
**Infer:** as in Session 4, the print statement only confirms the Python calls
returned without raising — independently checking both consoles is the only way
to be sure nothing (endpoint hours, BigQuery storage) is still quietly billing if
this cell was interrupted partway through.

## What to try next

* Add a scheduled trigger (a Cloud Scheduler job calling `pipeline_job.submit()`
  on a cron schedule) so this pipeline re-runs automatically as new rows land in
  the `sessions` BigQuery table, rather than being kicked off by hand — Session 14
  builds a related "retrain automatically, deploy only if better" pattern around
  Cloud Build instead of Cloud Scheduler.
* Compare `au_prc_threshold` values: rerun with the threshold set above what the
  first training run actually achieved, and confirm the `Condition` block
  correctly skips deployment — this is the cheapest way to prove the quality gate
  actually gates, rather than assuming it does from reading the code alone.
* Session 24 builds a similar conditional "quality gate" pattern for a CI/CD
  pipeline using GitHub Actions instead of Vertex AI Pipelines — worth comparing
  the two orchestration styles side by side.
* Try swapping `train_automl_model` for a custom `scikit-learn` training component
  (a plain `GradientBoostingClassifier`) to see how much of AutoML's result this
  simpler, much cheaper model can recover on a dataset this size.